# Limpeza dos dados


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from numpy.polynomial import Polynomial


df = pd.read_csv("ships.csv", encoding="latin1")

df.shape

## Limpeza na base de dados


In [ ]:
raw_cols = ["DWT", "LBP", "B", "D", "d", "Vs"]

n_before = len(df)
df = df[(df[raw_cols] != 0).all(axis=1)].reset_index(drop=True)
print(f"{n_before - len(df)} linhas removidas por conter zero em alguma coluna inicial ({len(df)} restantes)")

#df.to_csv("ships.csv", index=False, encoding="latin1")
df.head()


## Gráficos de dispersão


### Configurações dos gráficos

In [ ]:
label_dict = {
    "B": "Breadth [m]",
    "D": "Depth [m]",
    "LBP": "Length Between Perpendiculars [m]",
    "d": "Scantling Draft [m]",
    "DWT": "DWT [t]",
    "Vs": "Speed [m/s]",
}

SCATTER_PAIRS = [
    ("DWT", "LBP"),
    ("DWT", "B"),
    ("DWT", "D"),
    ("DWT", "d"),
    ("DWT", "Vs"),
    ("LBP", "B"),
    ("B", "D"),
    ("LBP", "D"),
    ("LBP", "L/B"),
    ("LBP", "B/D"),
    ("LBP", "L/D"),
    ("LBP", "d/D"),
]

# pares (x, y) em que a curva de regressao polinomial (DWT/LBP x dimensao) e mostrada
REGRESSION_PAIRS = {
    ("DWT", "LBP"), ("DWT", "B"), ("DWT", "D"), ("DWT", "d"), ("DWT", "Vs"),
    ("LBP", "B"), ("LBP", "D"),
}

# coluna -> chave usada nos dicionarios de restricao de porto (valor maximo permitido)
PORT_LIMIT_KEY_MAP = {"LBP": "L", "B": "B", "d": "d"}

# coluna -> chave usada no requisito de projeto
REQ_KEY_MAP = {"DWT": "DWT", "Vs": "Vs"}


RESTRICTION_COLORS = {
    "Suez": "tab:orange",
    "Panama": "tab:green",
    "St_Lawrence": "tab:purple",
    "Req_proj": "tab:red",
}


### Restrições


In [ ]:
Suez = {
    "B": 77.5,
    "L": 400,
    "d": 18.89
}

Panama = {
    "B": 32.31,
    "L": 289.6,
    "d": 12.04
}

St_Lawrence = {
    "B": 23.8,
    "L": 222.5,
    "d": 7.92
}

Req_proj = {
    "Vs": 15.3*0.5144,
    "DWT": 45100,
}

local_DWT=[35100, 55100]

PORT_LIMITS = {
    "Suez": Suez,
    "Panama": Panama,
    "St_Lawrence": St_Lawrence,
}


### Regressões TUD 2008


In [ ]:
def reg_LBP(DWT):
    return 41.647 * DWT**(0.133)


def reg_B(DWT):
    return min(15.04 + 0.000369 * DWT, 32.2)


def reg_D(DWT):
    return 9.69 + 0.000188 * DWT


def reg_d(DWT):
    return 7.41 + 0.000106 * DWT


# regressoes empiricas TUD 2008 (parametro em funcao do DWT)
TUD_REGRESSIONS = {
    "LBP": reg_LBP,
    "B": reg_B,
    "D": reg_D,
    "d": reg_d,
}

### Plots


In [ ]:
def poly_r2(x, y, degree):
    p = Polynomial.fit(x, y, degree)
    y_pred = p(x)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return p, 1 - ss_res / ss_tot


def best_polynomial(x, y, max_degree=10, r2_threshold=0.95):
    """menor grau (<= max_degree) que atinge r2_threshold; senao, o de maior R^2."""
    best_p, best_r2, best_degree = None, -np.inf, None
    for degree in range(1, max_degree + 1):
        p, r2 = poly_r2(x, y, degree)
        if best_p is None or r2 > best_r2:
            best_p, best_r2, best_degree = p, r2, degree
        if r2 >= r2_threshold:
            return p, r2, degree
    return best_p, best_r2, best_degree


def format_polynomial(p, var="x", precision=4):
    """escreve a funcao ajustada (coeficientes na escala original de var) como texto."""
    coeffs = p.convert(domain=[-1, 1], window=[-1, 1]).coef
    terms = []
    for power, c in enumerate(coeffs):
        if power == 0:
            terms.append(f"{c:.{precision}g}")
        elif power == 1:
            terms.append(f"{c:+.{precision}g}*{var}")
        else:
            terms.append(f"{c:+.{precision}g}*{var}^{power}")
    return "y = " + " ".join(terms)


def plot_scatter(sel_df: pd.DataFrame, reg=True):
    for x, y in SCATTER_PAIRS:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(sel_df[x], sel_df[y], color="steelblue",
                edgecolor="black", alpha=0.5, s=15)
        ax.set_xlabel(label_dict.get(x, x))
        ax.set_ylabel(label_dict.get(y, y))
        # ax.set_title(f"{x} x {y}")

        x_min, x_max = sel_df[x].min(), sel_df[x].max()
        y_min, y_max = sel_df[y].min(), sel_df[y].max()

        # restricoes de porto: linha tracejada no valor maximo permitido
        for name, limits in PORT_LIMITS.items():
            color = RESTRICTION_COLORS[name]
            x_key = PORT_LIMIT_KEY_MAP.get(x)
            y_key = PORT_LIMIT_KEY_MAP.get(y)
            if x_key in limits:
                ax.axvline(limits[x_key], color=color, linestyle="--", label=name)
                x_max = max(x_max, limits[x_key])
            if y_key in limits:
                ax.axhline(limits[y_key], color=color, linestyle="--", label=name)
                y_max = max(y_max, limits[y_key])

        # requisito de projeto: linha tracejada no valor alvo
        x_req_key = REQ_KEY_MAP.get(x)
        y_req_key = REQ_KEY_MAP.get(y)
        if x_req_key in Req_proj:
            ax.axvline(Req_proj[x_req_key], color=RESTRICTION_COLORS["Req_proj"],
                    linestyle="--", label="Req_proj")
            x_max = max(x_max, Req_proj[x_req_key])
        if y_req_key in Req_proj:
            ax.axhline(Req_proj[y_req_key], color=RESTRICTION_COLORS["Req_proj"],
                    linestyle="--", label="Req_proj")
            y_max = max(y_max, Req_proj[y_req_key])

        regression_eq = None
        if reg:
            # regressao polinomial (grau <= 10, buscando R^2 >= 0.95)
            if (x, y) in REGRESSION_PAIRS:
                x_data = sel_df[x].to_numpy(dtype=float)
                y_data = sel_df[y].to_numpy(dtype=float)
                p, r2, degree = best_polynomial(x_data, y_data)
                x_line = np.linspace(x_data.min(), x_data.max(), 200)
                ax.plot(
                    x_line, p(x_line), color="black", linewidth=2,
                    label=f"regressao grau {degree} (R²={r2:.3f})",
                )
                regression_eq = format_polynomial(p, var=x)

            # regressao empirica TUD 2008 (parametro em funcao do DWT)
            if x == "DWT" and y in TUD_REGRESSIONS:
                dwt_line = np.linspace(sel_df[x].min(), sel_df[x].max(), 200)
                tud_values = np.array([TUD_REGRESSIONS[y](v) for v in dwt_line])
                ax.plot(dwt_line, tud_values, color="yellow", linewidth=2,
                        linestyle="-.", label="Regressão TUD 2008")
                y_max = max(y_max, tud_values.max())

        handles, labels = ax.get_legend_handles_labels()
        if labels:
            by_label = dict(zip(labels, handles))
            ax.legend(by_label.values(), by_label.keys())

        plt.tight_layout()
        plt.show()

        if regression_eq:
            print(f"{x} x {y} -> {regression_eq}")


In [ ]:
plot_scatter(df)

In [ ]:
df_local = df[(df["DWT"] >= local_DWT[0]) & (df["DWT"] <= local_DWT[1])].reset_index(drop=True)
print(df_local.shape)

plot_scatter(df_local, reg=False)

### Análise estatística


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()


def mode_of(series):
    m = series.mode()
    return m.iloc[0] if not m.empty else np.nan


stats_rows = {}
for col in numeric_cols:
    s = df[col].dropna()
    mean = s.mean()
    std = s.std(ddof=1)
    stats_rows[col] = {
        "media": mean,
        "mediana": s.median(),
        "max": s.max(),
        "min": s.min(),
        "moda": mode_of(s),
        "faixa": s.max() - s.min(),
        "erro padrao": stats.sem(s),
        "variancia": s.var(ddof=1),
        "desv pad": std,
        "coef variancia": std / mean if mean != 0 else np.nan,
        "skewness": stats.skew(s),
        "kurtosis": stats.kurtosis(s),
    }

stats_df = pd.DataFrame(stats_rows).T
stats_df


In [ ]:
for col in numeric_cols:
    s = df[col].dropna()
    fig, (ax_hist, ax_box) = plt.subplots(1, 2, figsize=(10, 4))
 
    ax_hist.hist(s, bins=50, color="steelblue", edgecolor="black")
    ax_hist.set_xlabel(label_dict.get(col, col))
    ax_hist.set_ylabel("frequencia")
    ax_hist.set_title(f"Histograma - {col}")

    ax_box.boxplot(s, vert=True)
    ax_box.set_ylabel(label_dict.get(col, col))
    ax_box.set_title(f"Boxplot - {col}")
    ax_box.set_xticks([])

    plt.tight_layout()
    plt.show()
